In [3]:
import cv2
import face_recognition
import numpy as np
import sqlite3
import time
import os

# Database file
DB_FILE = 'face_recognition.db'

# Create SQLite database and table to store face encodings
def create_db():
    conn = sqlite3.connect(DB_FILE)
    c = conn.cursor()
    c.execute('''CREATE TABLE IF NOT EXISTS faces (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    name TEXT,
                    encoding BLOB)''')
    conn.commit()
    conn.close()

# Function to insert new face encodings into the database
def insert_face_encoding(name, encoding):
    conn = sqlite3.connect(DB_FILE)
    c = conn.cursor()
    encoding_bytes = encoding.tobytes()  # Convert encoding to bytes
    c.execute("INSERT INTO faces (name, encoding) VALUES (?, ?)", (name, encoding_bytes))
    conn.commit()
    conn.close()

# Capture images of the user from different angles
def capture_new_user_images(name, num_images=3):
    user_folder = f"dataset/{name}"
    if not os.path.exists(user_folder):
        os.makedirs(user_folder)

    print(f"Capturing images for {name}...")

    cap = cv2.VideoCapture(0)  # Start webcam capture
    angles = ['Left', 'Center', 'Right']
    frame_w, frame_h = 600, 500  # Size of the yellow frame

    # Loop through each angle for posing
    for angle in angles:
        print(f"Please pose {angle} inside the yellow rectangle.")

        captured = 0
        while captured < num_images:
            ret, frame = cap.read()
            if not ret:
                print("Failed to grab frame.")
                break

            # Get the current frame size
            frame_height, frame_width, _ = frame.shape

            # Calculate the center coordinates to place the yellow frame in the middle
            frame_x = int((frame_width - frame_w) / 2)
            frame_y = int((frame_height - frame_h) / 2)

            # Draw the yellow rectangle in the center
            cv2.rectangle(frame, (frame_x, frame_y), (frame_x + frame_w, frame_y + frame_h), (0, 255, 255), 2)

            # Display pose instruction on screen
            cv2.putText(frame, f"Pose {angle} inside the rectangle", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

            # Find all face locations in the current frame
            face_locations = face_recognition.face_locations(frame)
            
            for (top, right, bottom, left) in face_locations:
                # Check if the face is inside the yellow rectangle
                if (left > frame_x and right < frame_x + frame_w and top > frame_y and bottom < frame_y + frame_h):
                    # Save the image if the face is inside the rectangle
                    image_path = f"{user_folder}/{name}_{angle.lower()}_{captured + 1}.jpg"
                    cv2.imwrite(image_path, frame)
                    captured += 1
                    print(f"Captured {captured} images for {angle} pose.")

            # Display the frame
            cv2.imshow("Capturing...", frame)

            # Break the loop if 3 images have been captured or if the user presses 'q'
            if captured >= num_images or cv2.waitKey(1) & 0xFF == ord('q'):
                break

            # Wait until the user positions their face inside the rectangle
            if captured == 0:
                print(f"Position your face inside the yellow rectangle to start capturing {angle} pose.")

        # Wait a bit before moving to the next pose
        time.sleep(1)

    cap.release()
    cv2.destroyAllWindows()
    print(f"Captured {captured} images for {name}.")

# Load face encodings from the database
def get_all_face_encodings():
    conn = sqlite3.connect(DB_FILE)
    c = conn.cursor()
    c.execute("SELECT id, name, encoding FROM faces")
    rows = c.fetchall()

    known_face_encodings = []
    known_face_names = []
    for row in rows:
        name = row[1]
        encoding = np.frombuffer(row[2], dtype=np.float64)
        known_face_encodings.append(encoding)
        known_face_names.append(name)

    conn.close()
    return known_face_encodings, known_face_names

# Add new user to the database
def add_new_user():
    current_user = input("Enter your name for registration: ")

    # Capture the user's images
    capture_new_user_images(current_user)

    # Process the captured images and store face encodings in the database
    user_folder = f"dataset/{current_user}"
    for image_path in os.listdir(user_folder):
        image_path_full = os.path.join(user_folder, image_path)
        image = face_recognition.load_image_file(image_path_full)
        face_encodings = face_recognition.face_encodings(image)
        
        if face_encodings:
            # Save the encoding to the SQLite database
            insert_face_encoding(current_user, face_encodings[0])

    print(f"User {current_user} has been added to the database.")

# Recognize faces in real-time using the webcam
def recognize_faces():
    cap = cv2.VideoCapture(0)  # Start webcam capture

    # Load known face encodings from the database
    known_face_encodings, known_face_names = get_all_face_encodings()

    while True:
        ret, frame = cap.read()
        if not ret:
            print("Failed to grab frame.")
            break

        # Find face locations and encodings in the current frame
        face_locations = face_recognition.face_locations(frame)
        face_encodings = face_recognition.face_encodings(frame, face_locations)
        
        for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):
            matches = face_recognition.compare_faces(known_face_encodings, face_encoding)
            name = "Unknown"
            accuracy = 0.0

            if True in matches:
                first_match_index = matches.index(True)
                name = known_face_names[first_match_index]
                accuracy = 1.0  # You can improve accuracy by using distance comparison

            # Draw rectangle around the face and display the name and accuracy
            color = (0, 255, 0) if name != "Unknown" else (0, 0, 255)
            cv2.rectangle(frame, (left, top), (right, bottom), color, 2)
            cv2.putText(frame, f"{name} ({accuracy*100:.2f}%)", (left, top - 10),
                        cv2.FONT_HERSHEY_DUPLEX, 0.5, (255, 255, 255), 1)

        # Display the resulting frame
        cv2.imshow("Face Recognition", frame)

        # Break the loop on pressing 'q'
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

# Main entry point
def main():
    # Create the database if it doesn't exist
    create_db()

    # Choose between adding a new user or recognizing faces
    choice = input("Enter '1' to add new user or '2' to recognize faces: ")
    if choice == '1':
        add_new_user()
    elif choice == '2':
        recognize_faces()
    else:
        print("Invalid choice. Exiting...")

# Run the main function
main()


Enter '1' to add new user or '2' to recognize faces:  1
Enter your name for registration:  Khizra


Capturing images for Khizra...
Please pose Left inside the yellow rectangle.
Position your face inside the yellow rectangle to start capturing Left pose.
Captured 1 images for Left pose.
Captured 2 images for Left pose.
Captured 3 images for Left pose.
Please pose Center inside the yellow rectangle.
Position your face inside the yellow rectangle to start capturing Center pose.
Position your face inside the yellow rectangle to start capturing Center pose.
Position your face inside the yellow rectangle to start capturing Center pose.
Captured 1 images for Center pose.
Captured 2 images for Center pose.
Captured 3 images for Center pose.
Please pose Right inside the yellow rectangle.
Captured 1 images for Right pose.
Captured 2 images for Right pose.
Captured 3 images for Right pose.
Captured 3 images for Khizra.
User Khizra has been added to the database.
